# Task 16 — Series, Analysis, and Cleaning

This notebook is a practice worksheet, and it comes in three movements. The first four questions (Q-1 to Q-5) drill the Series fundamentals you met in the Series session: creating a Series from scratch, running arithmetic between two Series, comparing element by element, and changing a column's data type. The middle block (Q-6 to Q-8) turns to a real dataset — every batsman's total runs in IPL matches from 2008 to 2022, downloaded as one Series — and asks filtering questions against it. The final block (Q-9 and Q-10) loads an items price table from CSV and walks a complete cleaning pipeline: strip currency symbols, coerce types, patch missing values, then summarise and plot.

Each question is a markdown cell followed by a '# code here' cell. Your job is to replace the placeholder with working code and run it. The teaching intent behind each question matters more than the code itself:

- Q-1 tests the bare constructor — the empty series shows the skeleton of index, values, and dtype.
- Q-2 tests element-wise operators and index alignment — the moment pandas reuses + and - on whole columns.
- Q-3 tests comparison operators — the birth of boolean masks, the same masks used to filter data in Q-6, Q-7, and Q-8.
- Q-5 tests type conversion — astype for the simple cases, pd.to_numeric with errors='coerce' for messy columns.
- Q-6 to Q-8 test the mask-then-count recipe — nlargest and sort_values for leaderboards, (series > threshold).sum() for counts.
- Q-9 tests the cleaning chain — index_col, str.replace, to_numeric, fillna(mean()).
- Q-10 tests summary statistics — mean, quantile, plot.hist, and between.

Work strictly in order. Each question builds the muscle the next one uses, and the filters in the middle block are exactly the tools the cleaning block depends on. When you print a result, look at it: the count questions produce a single integer, the mask questions produce a boolean stream, and the cleaning block should end with a float column containing no NaN.

Do not move the code cells or delete the question text. Re-read each question before you code, and use the hints embedded in each theory block only when you are genuinely stuck. When every cell runs without error, save the file — you have just completed a full round of series manipulation from construction to statistics.

# 1. Creating an Empty Series

### Q-1: Write a program to create an empty series.

A Series object pairs two arrays: the VALUES and a matching INDEX of labels. Creating an empty Series is the usual first exercise because it exposes the constructor's skeleton before any data arrives. `pd.Series()` returns an empty Series — its dtype is object, its index is empty, and printing it shows the familiar two-line view plus `dtype: object`.

For most real pipelines an empty holder with an explicit type is more useful, because every future addition must then conform to it. `pd.Series(dtype='float64')` reserves an empty container of type float, `dtype='int64'` makes it integer, and `dtype=object` accepts values of any Python type. Passing a list as data seeds the Series with the default RangeIndex 0..n-1 and infers the dtype from the values.

The constructor reads its dtype from the data when none is given, which is why `pd.Series([1, 2, 3])` comes back as int64 while `pd.Series([1, 'a'])` is promoted to object. When you later see 'object' on a column that should hold numbers, you know exactly where it came from — and the castable access to to_numeric is one task down the road.

> Hint: `pd.Series(dtype=float)` in the code cell; confirm the output shows `dtype: float64` next to an empty index.
> Hint: `pd.Series()` with no arguments is the minimal empty series.

In [ ]:
# code here

# 2. Series Arithmetic

### Q-2: Write a Pandas program to add, subtract, multiply and divide two Pandas Series.

Pandas operators are element-wise: when you write `s1 + s2`, pandas aligns the two Series by INDEX, matches labels, and applies the operator to the pairs that exist on both sides. Give both Series the same default index 0..4 and the result contains five values computed point-by-point: 10+1, 20+3, 30+5, 40+7, 50+10.

The full operator set mirrors plain Python: `+` addition, `-` subtraction, `*` multiplication, `/` division. The notebook expects all four — assign each result to its own name and print it, so every one of the sum, difference, product, and quotient Series is visible.

Alignment is pandas' superpower and its sharpest gotcha. If the two indexes differ, pandas still aligns by label: labels found on only one side become NaN in the result. `fill_value=0` inside `s1.add(s2, fill_value=0)` substitutes a zero for any missing partner instead.

The named methods `.add()`, `.sub()`, `.mul()`, `.div()` exist alongside the operators and accept these extra arguments — alignment becomes controllable rather than a surprise.

> Hint: print the result of each operation separately so all four outputs are visible.
> Hint: `s1 + s2`, `s1 - s2`, `s1 * s2`, and `s1 / s2` — no loops needed.

In [ ]:
# code here

# 3. Comparing Two Series

### Q-3: Write a Pandas program to compare the elements of the two Pandas Series.
### Sample Series: [2, 4, 6, 8, 10], [1, 3, 5, 7, 10]

Comparison operators paste across pandas exactly like arithmetic: `s1 > s2`, `s1 < s2`, `s1 == s2` each return a boolean Series of the same length as the aligned inputs. Every element is tested independently, and the output reports a per-label verdict.

With the samples [2, 4, 6, 8, 10] and [1, 3, 5, 7, 10], there is no single answer like 'the first is bigger' — pandas produces a position-by-position report: at label 0, 2 > 1 is True; at 1, 4 > 3 is True; and so on until label 4, where 10 > 10 is False and 10 == 10 is True.

The result of any comparison is a MASK: a Series of True/False values that can itself be used to filter. `s1[s1 > 5]` keeps only the elements greater than 5. Boolean indexing — the foundation of every filter in this course — is nothing more than subscripting a Series with the mask a comparison produces.

Practical gotchas: `!=` tests inequality and `==` is the equality test within pandas; the identity keyword `is` is never used against elements. When values are missing, NaN never equals NaN, so `s1 == s1` can still show False where gaps exist.

> Hint: `s1 > s2` is the whole comparison; assign it to a variable and print.
> Hint: also print `s1 < s2`, `s1 == s2`, and `s1 >= s2` to demonstrate each operator.

In [ ]:
# code here

# 4. Changing the Data Type

### Q-5: Write a function to change the data type of a given column or a Series. The function takes the series and the data type as input and returns the converted series.
### series = pd.Series([1, 2, 'Python', 2.0, True, 100]) → convert to float

A single Series can mix integers, strings, floats, and booleans — pandas then stores it as dtype object, which blocks correct arithmetic. Converting requires a per-element cast, and `pd.to_numeric()` performs exactly that: it parses every element that can become a number and raises on the ones that cannot.

Your function should accept a series and a dtype argument, then return `series.astype(dtype)`. astype is the general conversion tool: `astype(int)`, `astype(float)`, `astype(str)` all exist. But the sample contains the word 'Python', which a float simply cannot represent — so `astype(float)` alone would crash. The note in the question points you to to_numeric, the tool that fails gracefully: `pd.to_numeric(series)` converts every convertible element and reports the rest, and with `errors='coerce'` it replaces the stubborn 'Python' with NaN instead of raising.

The key takeaway: astype forces a column-wide conversion that must succeed everywhere, while to_numeric with errors='coerce' is the tolerant 'convert what you can, mark the rest missing' version — the workhorse of real-world cleaning. The pandas dtype inventory to remember: int64, float64, bool, object, category, datetime64.

> Hint: return `pd.to_numeric(series, errors='coerce')`, or a try/except wrapper around `astype(float)`.
> Hint: wrap the conversion in a function that takes `series` and `dtype` parameters.

In [ ]:
# code here

# 5. The Batsman Runs Dataset

# 6. Q-6 to Q-8 — Filtering the Batsman Series

### Q-6: Find the top 10 most run getters from the series.

Now the questions switch from constructing Series to ANALYSING one. The downloaded data holds every batsman's total runs in IPL 2008-2022 as a single Series: each player's name is an index label and each value is that player's career run total.

Finding the top ten run-getters is the leaderboard question. Two idioms give the same answer: `batsman.sort_values(ascending=False).head(10)` sorts the whole Series descending by runs and takes the first ten rows — names on the left, totals in order — while `batsman.nlargest(10)` does it in a single call designed for exactly this task. Both keep the index labels attached, which is what makes the output readable as a leaderboard.

> Hint: `series.nlargest(10)` is the shortest correct answer.

### Q-7: Number of players having runs above 3000.

The count question nests a filter inside a count. `batsman > 3000` produces the boolean mask of players above the threshold; `(batsman > 3000).sum()` adds the True values (True counts as 1) and returns the number of qualifying players.

> Hint: `(series > 3000).sum()` counts the players in one line.

### Q-8: Number of players having runs above the mean value?

Replace the hard-coded threshold with the data's own reference point: `series > series.mean()` — the players outperforming the average batsman. Same mask-then-sum recipe.

> Hint: `(series > series.mean()).sum()` is your answer.

The trio teaches the fundamental pandas filter recipe — build a mask with a comparison, then SUM it for a count, or SUBSCRIPT the Series with it to see actual rows: `series[series > 3000]` prints every player above the bar while `(series > 3000).sum()` tells you how many there are.

In [ ]:
# code here

###`Q-7` No of players having runs above 3000

In [ ]:
# code here

###`Q-8` No of players having runs above mean value?

In [ ]:
# code here

# 7. The items Dataset

# 8. Q-9 — Cleaning a Price Series

### Q-9:
### i. Read items.csv making item_name the index.
### ii. Show the number of NaN values.
### iii. Item price is given in $, convert it to rupees WITHOUT the currency symbol.
### iv. Make the data type of the newly made series float.
### v. Fill NaN with the mean of the series.

This is the classic real-world mess: prices stored as '$2.39' inside a column — text, not numbers. Five staged riddles walk the whole cleaning pipeline.

First, reading with an index: `pd.read_csv('items.csv', index_col='item_name')` tells pandas which column becomes the row labels, so every later operation is addressed by item name.

Second, counting holes: `df['item_price'].isnull().sum()` — isnull builds the boolean mask of missing cells and sum() counts the True entries, showing exactly how many prices are NaN.

Third, the currency fight: the column holds text like '$2.39'. Use a vectorised string method — `series.str.replace('$', '', regex=False)` strips the symbol from every element at once — or a Python lambda: `series.apply(lambda x: str(x).replace('$', ''))`. Either way the symbol is gone.

Fourth, typing: with the symbol removed, every value looks like a number but pandas still holds text. Convert the column to real numbers with `pd.to_numeric(series)` or `astype(float)`, completing the rupees conversion from the raw money string.

Fifth, patching: `series.fillna(series.mean())` replaces every missing price with the column's average, leaving no NaN cells behind.

> Hint: chain — read_csv(index_col='item_name') → str.replace → pd.to_numeric → fillna(mean()); then print `.dtype` and `.isnull().sum()` to verify.

The five steps move a textual money column to a float column with zero gaps — precisely the readiness state data should carry into aggregation.

In [ ]:
# code here

# 9. Q-10 — Statistics and a Histogram

### Q-10:
### i. Find the mean price.
### ii. Find the 30th and the 6th percentile value.
### iii. Plot a histogram of price with bin size 50.
### iv. Number of items whose price lies between 1000 and 2000.

With a clean float price series, the final questions demand a full statistical look.

Central tendency is one call: `series.mean()` returns the average price — a single number summarising the whole column.

Percentiles divide the distribution; the famous one is the median at the 50th point. `series.quantile(0.3)` and `series.quantile(0.06)` return the 30th and 6th percentile values respectively — each is the price below which that percentage of items falls.

Histogram: `series.plot.hist(bins=50)` — one call, 50 bars, instantly revealing the distribution's shape, with most items piling into the low-price buckets and a long tail to the right.

The range filter is one boolean mask: `series[(series >= 1000) & (series <= 2000)]` — each chained condition is parenthesised, joined with &, and the outer brackets keep the passing rows; wrapping the whole mask in sum() counts them: `series.between(1000, 2000).sum()` does the same work in a single method.

> Hint: mean(), quantile(0.3), quantile(0.06), plot.hist(bins=50), then between(1000, 2000).sum().
> Hint: when combining two conditions, always wrap each comparison in its own parentheses.

## Summary

Task 16 pulled together everything the Series can do. You created empty Series, combined two with +, -, *, and /, compared them element by element, and converted mixed object columns into clean floats with to_numeric and astype. On the IPL batsman data you produced leaderboards with nlargest and sorted counts above thresholds, and above the mean, using masks and .sum(). On the items table you read a CSV with a chosen index, counted missing values, stripped currency text, converted the column type, and filled the gaps with the mean. You then summarised the series with mean and quantile, shaped its distribution with a histogram, and bounded a range with between. If the pipeline now feels like a recipe — read, clean, count, summarise, plot — that is precisely the pandas order of operations you will apply to every task that follows.

In [ ]:
# code here